# ScreenGuide: fine-tune Laya on Colab

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine; L4 or A100 is faster). Then Runtime → **Run all**.

What this notebook does:
1. Clones the public repo and installs Laya.
2. Fine-tunes the open Laya decision model on teacher-generated goal → menu-command data. The training rows contain no personal data.
3. Runs the 35-goal held-out test (Preview, Finder, System Settings, Safari). These apps are never seen in training.
4. Zips the checkpoint for download.

Training happens in the cloud; the **product runs the model locally** on the Mac. See `ml/FINETUNE.md` and `ml/RESULTS.md` in the repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf app_builders_hack && git clone -q https://github.com/EZA2405/app_builders_hack
%cd /content/app_builders_hack/ml
!pip -q install "laya[serve]"
!python -c "import laya, torch; print('laya', laya.__version__, '| cuda', torch.cuda.is_available())"
!wc -l data/train.jsonl data/val.jsonl

## Train
- **Main run:** `v1` (base checkpoint, 3 epochs). Prints validation accuracy before and after, on 4 apps never trained on.
- **Optional:** if time allows, run `v3` with 6 epochs.

In [ ]:
import time, subprocess, os
# ---- choose a run ----
VERSION = "v2"                      # name of the output model
DATA = "data/v2/train.jsonl"        # v2 hard labels; use "data/v2/train_soft.jsonl" for Jev soft labels
EPOCHS = 3
VAL = "data/v2/val.jsonl"
# ----------------------
t = time.time()
!laya-train --data {DATA} --eval {VAL} --base convaiinnovations/laya \
  --out /content/models/laya-guide-{VERSION} --device cuda --epochs {EPOCHS} \
  --micro-batch 8 --grad-accum 8 --shuffle-options --label-smoothing 0.05 --seed 7 2>&1 | tee /content/train_{VERSION}.log
print(f"training took {(time.time()-t)/60:.1f} min")

## Held-out test (35 goals, 4 unseen apps)
This starts a local Laya server with both the base model and the fine-tuned one, then runs `bench_localjev.py`, a stdlib-only script, through a 16-wide tournament.

In [ ]:
import os, json, time, subprocess, urllib.request
env = dict(os.environ, LAYA_HOST="127.0.0.1", LAYA_PORT="8766", LAYA_DEVICE="cuda",
           LAYA_EXTRA_MODELS=json.dumps({"guide": f"/content/models/laya-guide-{VERSION}"}))
server = subprocess.Popen(["laya-serve"], env=env, stdout=open("/content/serve.log","w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8766/health", timeout=2); break
    except Exception:
        time.sleep(2)
print("server up")

In [ ]:
import re, glob
def bench(model, fixtures, label):
    total = hits = 0
    for f in fixtures:
        out = subprocess.run(["python3", "bench_localjev.py", "--url", "http://127.0.0.1:8766", "--model", model,
                              "--group", "16", "--fixture", f], capture_output=True, text=True).stdout
        print(out.strip().splitlines()[-1] if out.strip() else out)
        m = re.search(r"accuracy (\d+)/(\d+)", out)
        if m: hits += int(m[1]); total += int(m[2])
    print(f"===== {model} {label}: {hits}/{total} =====")
    return hits, total

# Model selection: full-tournament VALIDATION (4 apps never trained on). v1 baseline = 79/127.
val = bench("guide", sorted(glob.glob("fixtures/val/*.json")), "VAL (full tournament)")

### Held-out test: run **only for the variant you pick** (by the VAL number above)
Report every held-out run you do, not just the best. v1 = 25/35, base = 8/35.

In [ ]:
held = bench("guide", [f"fixtures/{a}_real.json" for a in ["preview", "finder", "systemsettings", "safari"]], "HELD-OUT 35")
print("VAL", val, "| HELD-OUT", held)

## Download the checkpoint
Upload it to the team (Google Drive or AirDrop). **Don't commit weights to git.** Post the numbers above in the team chat, and add a row to `ml/RESULTS.md`.

In [ ]:
!cd /content/models && zip -qr /content/laya-guide-{VERSION}.zip laya-guide-{VERSION} && ls -lh /content/laya-guide-{VERSION}.zip
from google.colab import files
files.download(f"/content/laya-guide-{VERSION}.zip")